# Create table with jumps from all mutations and sites
final dataframe will be saved in `outputs/jumps.parquet`
### In order to create jump dataframe:
1. run graph creation for each site
2. add information about mutation
3. concatenate and save dataframes

-------
Then you can just filter nodes that are not jumping 
```py
df = df[df['jump_event'] == True]
```


In [10]:
import sys, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
SCRIPTS_DIR  = PROJECT_ROOT / 'scripts'
DATA_PATH    = PROJECT_ROOT / 'data' / 'single-cell-tracks_exp1-6_noErbB2.csv.gz'
META_PATH    = PROJECT_ROOT / 'data' / '01-readme-experiment-description_2022-04-05.csv'
OUTPUT_ROOT  = PROJECT_ROOT / 'analysis_outputs'
FINAL_OUTPUT_PATH = PROJECT_ROOT / 'outputs'
EXP_ID = 1
SIGNAL_COL = 'ERKKTR_ratio'
MAX_CPU_WORKERS = 8

In [2]:
meta_df = pd.read_csv(META_PATH)

In [3]:
def run_site(site, mutation=None):
    print(f"Starting graph computation for site {site}...")
    cmd = [
        sys.executable,
        str(SCRIPTS_DIR / 'spatiotemporal_signal_propagation.py'),
        '--data-path', str(DATA_PATH),
        '--meta-path', str(META_PATH),
        '--exp-id', str(EXP_ID),
        '--site-id', str(site),
        '--signal-col', SIGNAL_COL,
        '--output-dir', str(OUTPUT_ROOT),
    ]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return site, res

futures = []
with ThreadPoolExecutor(max_workers=MAX_CPU_WORKERS) as ex:
    for site, mutation in zip(meta_df['Site'], meta_df['Mutation']):
        futures.append(ex.submit(run_site, site, mutation))

for fut in as_completed(futures):
    site_id, res = fut.result()
    print(f"--- Output for site {site_id} ---")
    print(res.stdout)
    if res.returncode != 0:
        print(f"Error for site {site_id}:\n", res.stderr)

Starting graph computation for site 1...
Starting graph computation for site 2...
Starting graph computation for site 3...
Starting graph computation for site 4...
Starting graph computation for site 5...
Starting graph computation for site 6...
Starting graph computation for site 7...
Starting graph computation for site 8...
Starting graph computation for site 9...
Starting graph computation for site 10...
Starting graph computation for site 11...
Starting graph computation for site 12...
Starting graph computation for site 13...
Starting graph computation for site 14...
Starting graph computation for site 15...
Starting graph computation for site 16...
Starting graph computation for site 17...
Starting graph computation for site 18...
Starting graph computation for site 19...
Starting graph computation for site 20...
Starting graph computation for site 21...
Starting graph computation for site 22...
Starting graph computation for site 23...
Starting graph computation for site 24...
S

## Join dataframes

In [11]:
partial_dfs = []
for site, mutation in zip(meta_df['Site'], meta_df['Mutation']):
    try:
        df = pd.read_csv(OUTPUT_ROOT / f"exp_{EXP_ID}_site_{site}_ERKKTR_ratio/nodes.csv.gz")
    except FileNotFoundError:
        continue

    df['Site'] = site
    df['Mutation'] = mutation
    partial_dfs.append(df)
full_df = pd.concat(partial_dfs)

In [12]:
full_df.to_parquet(FINAL_OUTPUT_PATH / "jumps.parquet")

In [13]:
full_df

,node_id,Exp_ID,Image_Metadata_Site,track_id,Image_Metadata_T,time_h,objNuclei_Location_Center_X,objNuclei_Location_Center_Y,Nuclear_size,ERKKTR_ratio,...,signal_value,signal_delta,jump_event,neighbor_count,neighbor_jump_count,neighbor_jump_now,neighbor_mean_signal,future_self_jump,Site,Mutation
0,0,1,1,1,0,0.000000,932.211,875.248,303.000,0.704407,...,0.704407,NaN,False,22,0,False,0.927543,True,1,WT
1,1,1,1,1,1,0.083333,932.150,874.174,333.000,0.848242,...,0.848242,0.143835,True,22,7,True,0.963904,True,1,WT
2,2,1,1,1,2,0.166667,932.376,873.787,314.000,1.059170,...,1.059170,0.210928,True,22,5,True,1.004497,True,1,WT
3,3,1,1,1,3,0.250000,932.168,873.453,322.000,1.188000,...,1.188000,0.128830,True,22,8,True,1.029900,False,1,WT
4,4,1,1,1,4,0.333333,931.146,872.885,313.999,1.205540,...,1.205540,0.017540,False,23,4,True,1.049103,False,1,WT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251973,251973,1,20,1649,252,21.000000,165.727,715.734,128.000,0.526214,...,0.526214,-0.003472,False,13,0,False,0.866761,False,20,PTEN_del
251974,251974,1,20,1649,253,21.083333,165.758,715.422,128.000,0.539510,...,0.539510,0.013296,False,13,1,True,0.873478,False,20,PTEN_del
251975,251975,1,20,1649,254,21.166667,165.788,715.409,132.000,0.505667,...,0.505667,-0.033843,False,12,1,True,0.877968,False,20,PTEN_del
251976,251976,1,20,1649,256,21.333333,165.888,715.480,125.000,0.515036,...,0.515036,0.009369,False,12,1,True,0.872689,False,20,PTEN_del
